# Notebook 07 — Code-Switching Deep Dive

The interesting failure case in voice. "Code-switching" is when a speaker mixes languages mid-utterance — *"明天的 standup 我们 review 一下 Q3 的 OKR"*. It's the dominant speech pattern in tech offices in mainland China, Hong Kong, Singapore, India, and bilingual professional contexts everywhere.

Whisper was not specifically trained for this. It still works *kind of*. The point of this notebook is to **measure how badly it fails out of the box**, then walk through three mitigation strategies with numbers.

## What we'll do

1. Build a 5-sample eval set covering different code-switching patterns + ground-truth transcripts.
2. Build a tiny eval harness — overall **CER** (character error rate) + **English word recall** broken out separately.
3. Run three strategies head-to-head:
   - **A. Vanilla** — defaults, auto language detection. Baseline.
   - **B. Force `language=zh` + `initial_prompt`** — bias the decoder with the English glossary.
   - **C. VAD-chunked + per-chunk language detection** — let Whisper switch context per VAD segment.
4. Compare. Decide which to deploy.

## Hardware

CPU is fine. Total runtime ~3–5 minutes after the first model load.

## 1. Build the eval set

Five samples chosen to stress different code-switching patterns. Generated via OpenAI TTS so the lab is reproducible — but you should re-run with your own recordings to see realistic numbers (TTS produces clearer speech than humans).

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(dotenv_path="../.env")
client = OpenAI()

AUDIO_DIR = Path("../data/audio_samples/codeswitch")
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

# (id, text, expected English tokens to recall)
EVAL_SET = [
    (
        "cs1_standup",
        "明天的 standup 我们 review 一下 Q3 的 OKR，然后 sync 一下 backlog。",
        ["standup", "review", "Q3", "OKR", "sync", "backlog"],
    ),
    (
        "cs2_codereview",
        "这个 function 的 time complexity 是 O of n squared，需要 optimize 成 O of n log n。",
        ["function", "time", "complexity", "optimize"],
    ),
    (
        "cs3_meeting",
        "我们 schedule 一个 sync，把 deliverables 和 timeline 都 align 一下。",
        ["schedule", "sync", "deliverables", "timeline", "align"],
    ),
    (
        "cs4_casual",
        "这个 weekend 我们 grab 个 coffee 然后 hang out 一下吧。",
        ["weekend", "grab", "coffee", "hang", "out"],
    ),
    (
        "cs5_reverse",
        "Let's discuss the 重点 of this 项目, then we'll align on the 方案。",
        ["Let", "discuss", "of", "this", "then", "we", "align", "on", "the"],
    ),
]

def make(name, text, voice="alloy"):
    p = AUDIO_DIR / f"{name}.mp3"
    if p.exists():
        return p
    client.audio.speech.create(model="tts-1", voice=voice, input=text).stream_to_file(str(p))
    return p

audio_paths = {name: make(name, text) for name, text, _ in EVAL_SET}
print(f"{len(audio_paths)} samples ready in {AUDIO_DIR}")

## 2. The eval harness

Two metrics, each measuring something different:

- **CER** (character error rate) over the whole string. Works for both Chinese and English (each character a token). Lower is better.
- **English word recall** — what fraction of the expected English tokens appear in the transcript. Specifically catches the failure mode where Whisper translates English to Chinese.

Why two metrics? CER alone hides the worst code-switching failure. If Whisper transcribes *"明天的 standup"* as *"明天的站等"* (translation), CER barely changes (3 chars edit) but **the English semantic content is gone**. English-word recall catches that.

In [ ]:
import re
from jiwer import cer

def normalize(s: str) -> str:
    """Lowercase, strip punctuation, collapse whitespace. Apply to both ref and hyp."""
    s = s.lower()
    s = re.sub(r"[^\w\s\u4e00-\u9fff]", " ", s)   # keep CJK + alphanumerics
    return re.sub(r"\s+", " ", s).strip()

def english_recall(transcript: str, expected: list[str]) -> float:
    t = normalize(transcript)
    found = sum(1 for w in expected if re.search(rf"\b{re.escape(w.lower())}\b", t))
    return found / len(expected) if expected else 1.0

def score(transcript: str, ref: str, expected: list[str]) -> dict:
    return {
        "cer": round(cer(normalize(ref), normalize(transcript)), 3),
        "en_recall": round(english_recall(transcript, expected), 2),
    }

# Sanity check
print(score("明天的站等我们看一下季度的目标", EVAL_SET[0][1], EVAL_SET[0][2]))
# Should show high CER (text changed a lot) and en_recall=0 (no English words preserved)

## 3. Load the model and define the three strategies

In [ ]:
from faster_whisper import WhisperModel

asr = WhisperModel("large-v3", device="cpu", compute_type="int8")
print("loaded")

In [ ]:
# Strategy A — vanilla
def transcribe_vanilla(path: Path) -> str:
    segments, _ = asr.transcribe(str(path))
    return " ".join(s.text.strip() for s in segments)

# Strategy B — force language=zh + initial_prompt with English glossary biasing
GLOSSARY = (
    "standup, review, sync, backlog, OKR, Q1, Q2, Q3, Q4, function, complexity, "
    "optimize, schedule, deliverables, timeline, align, weekend, coffee, project, "
    "discuss, focus."
)
def transcribe_biased(path: Path) -> str:
    segments, _ = asr.transcribe(
        str(path),
        language="zh",
        initial_prompt=GLOSSARY,
    )
    return " ".join(s.text.strip() for s in segments)

# Strategy C — VAD-chunked, redetect language per chunk
# faster-whisper has a built-in VAD filter (Silero) — when on, it splits silence-bounded
# segments and re-runs language detection per segment.
def transcribe_vad_chunks(path: Path) -> str:
    segments, _ = asr.transcribe(
        str(path),
        vad_filter=True,
        vad_parameters={"min_silence_duration_ms": 200},
        condition_on_previous_text=False,   # don't lock to whatever was decoded earlier
    )
    return " ".join(s.text.strip() for s in segments)

STRATEGIES = {
    "A_vanilla": transcribe_vanilla,
    "B_biased":  transcribe_biased,
    "C_vad":     transcribe_vad_chunks,
}

## 4. Run the matrix

In [ ]:
results = {}
for sample_id, ref, expected in EVAL_SET:
    path = audio_paths[sample_id]
    results[sample_id] = {"ref": ref, "expected_en": expected, "runs": {}}
    for strategy_name, fn in STRATEGIES.items():
        hyp = fn(path)
        results[sample_id]["runs"][strategy_name] = {"hyp": hyp, **score(hyp, ref, expected)}

# Print per-sample
for sample_id, data in results.items():
    print(f"\n=== {sample_id} ===")
    print(f"REF:  {data['ref']}")
    for strat, run in data["runs"].items():
        print(f"  [{strat}]  cer={run['cer']:.2f}  en_recall={run['en_recall']:.2f}")
        print(f"           {run['hyp']}")

In [ ]:
# Aggregate — average across samples
from statistics import mean

summary = {}
for strat in STRATEGIES:
    cers = [r["runs"][strat]["cer"] for r in results.values()]
    recs = [r["runs"][strat]["en_recall"] for r in results.values()]
    summary[strat] = {"cer": round(mean(cers), 3), "en_recall": round(mean(recs), 3)}

print(f"{'strategy':<14} {'avg CER':>10} {'avg en-recall':>16}")
print("-" * 42)
for strat, s in summary.items():
    print(f"{strat:<14} {s['cer']:>10.3f} {s['en_recall']:>16.2f}")

## 5. Reading the table

Typical numbers we see (your run will vary slightly):

- **A vanilla**: high English recall on Chinese-dominant samples, but `cs5_reverse` (English-dominant frame) often goes badly — Whisper detects `en` and tries to translate the Chinese.
- **B biased** (forced `zh` + glossary): English recall jumps on Chinese-dominant samples because the glossary in `initial_prompt` keeps those tokens in the decoder's beam. But it *hurts* on the reverse sample (forcing `zh` makes the English frame look wrong).
- **C VAD-chunked**: best on heterogeneous samples — each chunk gets its own language detection, so the model can switch context. Worst on the shortest samples — fewer chunks means redetection is unreliable.

**No single strategy wins on every sample.** That's why production code-switching systems are usually:

1. **Detect the dominant language** of the file (or session).
2. If the dominant language is Chinese-with-English-jargon (the common case for Chinese tech offices), use **B**.
3. For unknown / mixed-frame audio (longer recordings, conversations), use **C**.
4. Always log the language detection result with confidence — when confidence is low, that's your signal to escalate to a more expensive model (SeamlessM4T, GPT-4o audio, or human review).

## 6. The escape hatch — large multilingual STT models

If your code-switching numbers don't pass the bar even with the strategies above, the next stop is a model that was *trained* with code-switching in mind:

- **SeamlessM4T-v2** (Meta) — handles code-switching natively across 96 languages. Heavier (~2 B params).
- **GPT-4o audio** — closed-source but excellent on code-switched Chinese-English.
- **Gemini 2.5** native audio — long-context audio understanding, but Google-only.

Tradeoff: latency and per-minute cost climb fast. Faster-Whisper + the right strategy is usually enough for tech-office code-switching.

## 7. Try with your own voice

TTS audio is *unrealistically clean*. To see the realistic numbers, record yourself reading the same five sentences (your phone's voice memo app, then drop the files into `data/audio_samples/codeswitch/cs1_standup.mp3` etc, replacing the TTS samples). Re-run the matrix.

Most students see CER 2–4× higher on real recordings, with the gap between strategies *widening*. The strategy choice matters more on hard inputs.

## What we learned

- Code-switching is a real production problem and Whisper handles it badly out of the box. Default settings can silently translate or drop English tokens.
- **Two metrics, not one** — overall CER hides the loss-of-English-content failure mode.
- Three cheap mitigations (force language + glossary, VAD chunking) recover most of the gap. Pick by audio profile.
- When the cheap mitigations aren't enough: SeamlessM4T-v2 or GPT-4o audio.

**Next:** [Notebook 08 — Voice Agent](08_voice_agent.ipynb). We wire STT into the full sequential pipeline: Whisper → GPT-4o (with tool calls) → OpenAI TTS, and measure the latency budget that determines whether the agent feels broken to a human.